# Model training 

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.metrics import (classification_report, confusion_matrix, log_loss,
                             balanced_accuracy_score, matthews_corrcoef)

In [ ]:
data = pd.read_feather("../data/less_features.feather")

In [ ]:
data

In [ ]:
results = pd.read_feather("../data/final_data_ny_big.feather")

In [ ]:
y_ny = results[["y_ny"]]
y_ny

In [ ]:
y = y_ny.iloc[:, 0].loc[data.index].astype(int)  # flatten to a Series, aligned on dates

# chronological 80/20 split (same as 3_Feature_Selection), no shuffling
split = int(len(data) * 0.8)
X_train, X_test = data.iloc[:split], data.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [ ]:
# C picked by time-series CV: each fold trains on the past, validates on the next block
model = make_pipeline(
    StandardScaler(),
    LogisticRegressionCV(Cs=np.logspace(-4, 2, 30), cv=TimeSeriesSplit(n_splits=5),
                         scoring="roc_auc", max_iter=10000),
).fit(X_train, y_train)

In [ ]:
model 

In [ ]:
lr = model[-1]
p_test = model.predict_proba(X_test)[:, 1]
print(f"best C:   {lr.C_[0]:.4f}")
print(f"CV AUC:   {lr.scores_[1].mean(axis=0).max():.3f}")
print(f"test AUC: {roc_auc_score(y_test, p_test):.3f}")
print(f"test acc: {accuracy_score(y_test, p_test > 0.5):.3f}   vs always-up baseline {y_test.mean():.3f}")
pd.Series(lr.coef_[0], index=data.columns).sort_values()

## Recommendations to improve the model

**Short version:** the code is fine. No data from the future leaks in, and the overnight features stop at 09:00. The problem is *how we measure the model* and *what we ask it to predict*.

---

### 1. The CV AUC of 0.60 is too optimistic
**What happened:** in `3_Feature_Selection` we tried many feature sets, kept the best one (k=39 plus the sign filter), and then reported the score **on the same folds we used to choose it**. That is like picking the best of 100 coin-flippers and calling them skilled. The score looks good partly because we went looking for a good score.

**Evidence:** on data the selection never saw, the test AUC is **0.500**, which is pure chance. The full L1 model without the selection gets a CV AUC of about 0.52. Those are the honest numbers.

**Fix:** choose features using only the training part of each fold (nested CV), or keep a separate validation period between train and test just for making choices. Don't choose k by looking at the test AUC either, because then the test set is no longer unseen.

### 2. Remove the 92 lagged single-stock returns
With only 722 training days and around 100 features, the model will find patterns that are just luck. "ADI's return yesterday predicts the NASDAQ today" is almost certainly one of those. Use a small set of features that each have an economic reason to matter:

`ovn_ret, ovn_vol, ovn_range, vxn, vix, ndx_ret_ny_prev, ndx_ret_1d, ndx_vol_20d, asia_ret, europe_ret, prev_ny_high_dist, prev_ny_low_dist, pct_up, Dayofweek`

### 3. Change the target (the biggest improvement)
Predicting **direction** (up or down) is very hard, close to impossible. Predicting **how big the move will be** is much easier, because volatility clusters: calm days tend to follow calm days and wild days follow wild days.

Results with the 14 features above:

| What we predict | Model | CV AUC | Test AUC | Test 95% CI |
|---|---|---|---|---|
| Up or down | Logistic regression | 0.508 | 0.575 | [0.49, 0.67] |
| Up or down | Gradient boosting | 0.491 | 0.544 | [0.46, 0.64] |
| **Big move or not** (\|ret\| > median) | **Logistic regression** | **0.613** | **0.660** | **[0.56, 0.74]** |

Only the "big move" model has a confidence interval that stays above 0.5, so it is the only result we can show is better than chance.

Options:
- predict the size of the move (`|ret_ny|` or the intraday range) with a regression
- keep predicting direction, but drop days with tiny moves (those labels are basically random)
- two steps: first predict whether the move will be big, then predict direction only on big-move days

### 4. More data
The test set has only 144 days, so any test AUC is uncertain by about ±0.09. With that much uncertainty, 0.50 and 0.57 can't be told apart. The NQ overnight data only starts in 2023, but the daily yfinance features (VIX, yields, NDX lags) go back about 20 years, which is roughly 5,000 days. Train a model on daily features only using that long history, then check whether adding the overnight features improves it on 2023 onward.

### 5. Report results more honestly
- **Report a confidence interval** for the test AUC (bootstrap the test days), so readers can see how much noise there is.
- **Compare log loss, not accuracy.** The market went up 57% of the time, so "always predict up" already scores 57% accuracy. Our 53% is worse than that. Log loss compared with simply predicting the base rate is a fairer test.
- **Walk-forward testing:** retrain every month on all past data and predict the next month, instead of a single 80/20 split. This is closer to how the model would actually be used.

---

**Priority:** do **1** and **3**. "We can predict volatility but not direction, and here is properly validated evidence" is a stronger project than chasing an AUC of 0.52 on direction.
